# BLUESTOCK FINTECH: MUTUAL FUND ANALYTICS PLATFORM
## Notebook 02: Data Cleaning, Standardization & SQL Database Loading
**Database:** SQLite (`db/bluestock_mf.db`)
**Schema:** Star-Schema Relational Data Warehouse


### 1. Cleaning Requirements & Transformation Principles
- Clean string fields and standardise categories.
- Deduplicate transaction records.
- Enforce strictly positive NAV and transaction values.
- Generate a time dimension `dim_date` covering market trading calendar.
- Calculate within-scheme daily returns without cross-scheme data leakage.


In [ ]:
import os
import sys
import sqlite3
import pandas as pd
sys.path.insert(0, os.path.abspath('..'))
from scripts.data_cleaning import clean_all_datasets
from scripts.etl_pipeline import initialize_database, load_tables_to_db, verify_database_counts


### 2. Execute Data Cleaning Pipeline


In [ ]:
cleaned_dict = clean_all_datasets()
for name, df in cleaned_dict.items():
    print(f'Processed {name:<28}: {len(df):>6} clean records')


### 3. Initialize SQLite Star Schema (`sql/schema.sql`)


In [ ]:
initialize_database()
load_tables_to_db(cleaned_dict)
verify_database_counts()


### 4. Verification SQL Queries


In [ ]:
conn = sqlite3.connect('../db/bluestock_mf.db')
q = '''
SELECT f.category, COUNT(*) as scheme_count, ROUND(AVG(f.expense_ratio_pct), 2) as avg_ter
FROM dim_fund f
GROUP BY f.category
ORDER BY avg_ter ASC;
'''
display(pd.read_sql_query(q, conn))
conn.close()
